
# Lab 1: Creating TMSIS Silver Layer

In this notebook, we will apply transformations from the bronze layer of the TMSIS data to create the silver layer.

## Objectives
1. **Enhance Bronze Reviews**: Learn how to import and preview medical center data from SQL Server using Lakehouse Federation.
2. **Data Transformation**: Add a categorical column based on the numerical value of the review
3. **Create a New Table**: Create a new table in the silver layer to store the data.

Let's get started!

####Setup
First step in all the labs will be to run a setup notebook. This notebook to create a catalog to store data and to set your default catalog 

In [ ]:
%run "../Lab 0 - Setup/SetCatalogName"

Run the SQL query below. From the column names it appears several date columns are being selected but numbers are being returned.
The data was receive as unix timestamps and we will convert them to date/time to make them easier to use.

In [ ]:
%sql
select adjdctn_dt, admission_dt,birth_dt, mdcd_pd_dt,srvc_bgnng_dt,srvc_endg_dt,dschrg_dt
from bronze.tmsis_claims

In the SQL query two functions are used. The first function from_unixtime takes a number and returns a date as a string. We want to store the value as an actual date not a string so the function to_timestamp is used to convert the string into a timestamp data type.

A new column CLAIM_RCV_DT will also be added to store the current timestamp on when the data was added to the table

In [ ]:
%sql
select * except (adjdctn_dt,admission_dt,birth_dt,mdcd_pd_dt,srvc_bgnng_dt,srvc_endg_dt,dschrg_dt),
  to_timestamp(from_unixtime(dschrg_dt/1000)) as DSCHRG_DT, 
  to_timestamp(from_unixtime(adjdctn_dt/1000)) as ADJDCTN_DT, 
  to_timestamp(from_unixtime(admission_dt/1000)) as ADMISSION_DT,
  to_timestamp(from_unixtime(birth_dt/1000)) as BIRTH_DT,
  to_timestamp(from_unixtime(mdcd_pd_dt/1000)) as MDCD_PD_DT,
  to_timestamp(from_unixtime(srvc_bgnng_dt/1000)) as SRVC_BGNNG_DT,
  to_timestamp(from_unixtime(srvc_endg_dt/1000)) SRVC_ENDG_DT,
  current_timestamp as CLAIM_RCV_DT
from bronze.tmsis_claims

Using the SQL statement above we are going to create or replace a table but with a slight twist.
Databricks offers many performance optimizations to make querying very large datasets fast called liquid clustering.

**Databricks Liquid Clustering**: Automatically optimizes data layout and file sizes for efficient querying. Simply specifiy which columns will frequently be used in WHERE or JOIN clauses and Databricks takes care of the rest.

In [ ]:
%sql
--Complete the beginning of the SQL clause to create a table. The table name has been provided for you
 silver.tmsis_claims
as
select * except (adjdctn_dt,admission_dt,birth_dt,mdcd_pd_dt,srvc_bgnng_dt,srvc_endg_dt,dschrg_dt),
  to_timestamp(from_unixtime(dschrg_dt/1000)) as DSCHRG_DT, 
  to_timestamp(from_unixtime(adjdctn_dt/1000)) as ADJDCTN_DT, 
  to_timestamp(from_unixtime(admission_dt/1000)) as ADMISSION_DT,
  to_timestamp(from_unixtime(birth_dt/1000)) as BIRTH_DT,
  to_timestamp(from_unixtime(mdcd_pd_dt/1000)) as MDCD_PD_DT,
  to_timestamp(from_unixtime(srvc_bgnng_dt/1000)) as SRVC_BGNNG_DT,
  to_timestamp(from_unixtime(srvc_endg_dt/1000)) SRVC_ENDG_DT,
  current_timestamp as CLAIM_RCV_DT
from bronze.tmsis_claims
cluster by DSCHRG_DT,ADMISSION_DT,CLAIM_RCV_DT

####Congratulations on Completing the Notebook